In [ ]:
import pandas as pd, sys
print(pd.__version__, sys.executable)

In [ ]:
from pathlib import Path
import pandas as pd

# Folder where the raw data lives. ".." means "go up one folder from notebooks/".
RAW_DATA_FOLDER = Path("../data/raw")

# The names of the three cart folders.
CART_FOLDER_NAMES = [
    "NMDID-cart-2960-meta",
    "NMDID-cart-2966-meta",
    "NMDID-cart-2975-meta",
]

# Go through each cart and print the names of its CSV files.
for cart_folder_name in CART_FOLDER_NAMES:
    cart_folder_path = RAW_DATA_FOLDER / cart_folder_name
    csv_file_paths = sorted(cart_folder_path.glob("*.csv"))

    print("Cart folder:", cart_folder_name)
    print("Number of CSV files:", len(csv_file_paths))
    for csv_file_path in csv_file_paths:
        print("   ", csv_file_path.name)
    print()

In [ ]:
# The 8 table names (same in every cart).
TABLE_NAMES = [
    "decedents",
    "medical-history",
    "medications",
    "substance-usages",
    "env-history",
    "distinctions",
    "ancestry",
    "chromosomal-abnormalities",
]

# A dictionary to hold every table. Key = (cart number, table name).
all_tables = {}

for cart_folder_name in CART_FOLDER_NAMES:
    # "NMDID-cart-2960-meta" -> "2960"
    cart_number = cart_folder_name.split("-")[2]

    for table_name in TABLE_NAMES:
        csv_path = RAW_DATA_FOLDER / cart_folder_name / f"{table_name}.csv"

        # dtype=str keeps everything as text, so nothing is changed by accident
        # (for example, a leading zero in a zip code would be lost as a number).
        # keep_default_na=False stops pandas from silently turning words like
        # "NA" or "None" into missing values. We want to decide that ourselves.
        table = pd.read_csv(csv_path, dtype=str, keep_default_na=False)

        # Remember where each row came from.
        table["source_cart"] = cart_number

        all_tables[(cart_number, table_name)] = table

# Print a size summary: rows and columns of each table.
summary_rows = []
for (cart_number, table_name), table in all_tables.items():
    summary_rows.append({
        "cart": cart_number,
        "table": table_name,
        "rows": table.shape[0],
        "columns": table.shape[1] - 1,  # minus our added source_cart column
    })

size_summary = pd.DataFrame(summary_rows)
print(size_summary.to_string(index=False))

In [ ]:
# Look at the structure of each table in cart 2960.
for table_name in TABLE_NAMES:
    table = all_tables[("2960", table_name)]

    print("=" * 70)
    print("TABLE:", table_name)
    print("Columns:", list(table.columns))
    print()
    # Show the first 3 rows. T flips the table so long rows are easy to read.
    print(table.head(3).T.to_string())
    print()

In [ ]:
# Stack the 3 decedents tables into one table, just for looking (not de-duplicated yet).
decedents_all_carts = pd.concat(
    [all_tables[(cart, "decedents")] for cart in ["2960", "2966", "2975"]],
    ignore_index=True,
)

# For each column, count blanks and distinct values.
dictionary_rows = []
for column_name in decedents_all_carts.columns:
    column_values = decedents_all_carts[column_name]
    blank_count = (column_values.str.strip() == "").sum()
    dictionary_rows.append({
        "column": column_name,
        "blank_percent": round(100 * blank_count / len(column_values), 1),
        "distinct_values": column_values.nunique(),
        "example_values": list(column_values[column_values.str.strip() != ""].unique()[:3]),
    })

decedents_dictionary = pd.DataFrame(dictionary_rows)
print(decedents_dictionary.to_string(index=False))

In [ ]:
# 1) Every label value with its count (all 1,500 rows, duplicates still included).
print("LABEL VALUES (all rows)")
print(decedents_all_carts["primary_cause_of_death"].value_counts(dropna=False).to_string())
print()

# 2) Label by cart: rows = label, columns = cart.
print("LABEL BY CART")
label_by_cart = pd.crosstab(
    decedents_all_carts["primary_cause_of_death"],
    decedents_all_carts["source_cart"],
)
print(label_by_cart.to_string())
print()

# 3) Value counts of the small categorical columns.
small_categorical_columns = [
    "sex_code",
    "decomposition_score",
    "settings",
    "identification_method_code",
    "marital_status_code",
    "gender_code",
]
for column_name in small_categorical_columns:
    print("COLUMN:", column_name)
    print(decedents_all_carts[column_name].value_counts(dropna=False).to_string())
    print()

In [ ]:
# The long tables and their "value" columns (the columns that hold the real information).
long_table_value_columns = {
    "medical-history": ["medical_history_type", "medical_history_detail"],
    "medications": ["drug_number"],
    "substance-usages": ["substance_usage_type"],
    "env-history": ["environmental_factor_type", "environmental_factor_item"],
    "distinctions": ["race", "ethnicity", "tribe"],
    "ancestry": ["ancestor", "birthplace", "city_of_birth", "state_of_birth", "country_of_birth"],
    "chromosomal-abnormalities": ["chromosomal_abnormalities_type", "abnormality_factor", "is_familial"],
}

for table_name, value_columns in long_table_value_columns.items():
    # Stack the 3 carts of this table (not de-duplicated yet).
    stacked = pd.concat(
        [all_tables[(cart, table_name)] for cart in ["2960", "2966", "2975"]],
        ignore_index=True,
    )
    # A row is "empty" if all its value columns are blank.
    is_empty_row = (stacked[value_columns].apply(lambda col: col.str.strip()) == "").all(axis=1)

    print("TABLE:", table_name)
    print("  total rows:", len(stacked))
    print("  empty rows:", int(is_empty_row.sum()))
    print("  rows with real values:", int((~is_empty_row).sum()))
    print("  distinct people with real values:", stacked.loc[~is_empty_row, "id"].nunique())
    for column_name in value_columns:
        print("  top values of", column_name, ":", stacked[column_name].value_counts().head(4).to_dict())
    print()

In [ ]:
# 1) Is the id <-> deidentified_record_number link one-to-one?
id_pairs = decedents_all_carts[["id", "deidentified_record_number"]].drop_duplicates()
print("Distinct (id, record number) pairs:", len(id_pairs))
print("Distinct ids:", id_pairs["id"].nunique())
print("Distinct record numbers:", id_pairs["deidentified_record_number"].nunique())
print()

# 2) Find the duplicate people (same id in more than one cart).
duplicated_ids = decedents_all_carts.loc[
    decedents_all_carts["id"].duplicated(keep=False), "id"
].unique()
print("People appearing in more than one cart:", len(duplicated_ids))

# 3) For duplicate people: in which columns do the copies disagree?
duplicate_rows = decedents_all_carts[decedents_all_carts["id"].isin(duplicated_ids)]
columns_to_compare = [c for c in decedents_all_carts.columns if c != "source_cart"]

disagreement_counts = {}
for column_name in columns_to_compare:
    distinct_per_person = duplicate_rows.groupby("id")[column_name].nunique()
    disagreement_counts[column_name] = int((distinct_per_person > 1).sum())

disagreements = pd.Series(disagreement_counts)
print("Columns where duplicate copies disagree (number of people):")
print(disagreements[disagreements > 0].to_string() if (disagreements > 0).any() else "None. All copies are identical.")

In [ ]:
# For each long table, compare the rows of duplicate people across carts.
for table_name, value_columns in long_table_value_columns.items():
    stacked = pd.concat(
        [all_tables[(cart, table_name)] for cart in ["2960", "2966", "2975"]],
        ignore_index=True,
    )
    duplicate_person_rows = stacked[stacked["id"].isin(duplicated_ids)]

    # Build one "signature" per (person, cart): the sorted set of its rows, written as text.
    signature_per_person_and_cart = (
        duplicate_person_rows
        .assign(row_text=duplicate_person_rows[value_columns].agg("|".join, axis=1))
        .groupby(["id", "source_cart"])["row_text"]
        .apply(lambda rows: " || ".join(sorted(rows)))
        .reset_index()
    )

    # For each person, count how many different signatures they have across carts.
    different_signatures = signature_per_person_and_cart.groupby("id")["row_text"].nunique()
    people_with_disagreement = int((different_signatures > 1).sum())

    print(
        table_name,
        "| duplicate people with rows in this table:", different_signatures.shape[0],
        "| people whose rows differ between carts:", people_with_disagreement,
    )

In [ ]:
OUTPUT_TABLES_FOLDER = Path("../outputs/tables")
OUTPUT_TABLES_FOLDER.mkdir(parents=True, exist_ok=True)

# 1) Save the decedents data dictionary (from Step 1d).
decedents_dictionary.to_csv(OUTPUT_TABLES_FOLDER / "01_decedents_data_dictionary.csv", index=False)

# 2) Save the table size summary (from Step 1b).
size_summary.to_csv(OUTPUT_TABLES_FOLDER / "01_table_size_summary.csv", index=False)

# 3) Save the label counts by cart (from Step 1e).
label_by_cart.to_csv(OUTPUT_TABLES_FOLDER / "01_label_by_cart.csv")

print("Saved files:")
for saved_file in sorted(OUTPUT_TABLES_FOLDER.glob("01_*.csv")):
    print("   ", saved_file.name)

In [ ]:
# 1) For each person, record ALL carts they appeared in, e.g. "2960" or "2960;2966".
carts_per_person = (
    decedents_all_carts.groupby("id")["source_cart"]
    .apply(lambda carts: ";".join(sorted(carts)))
    .rename("source_carts")
)

# 2) Keep ONE row per id. All copies are identical (checked in Step 1g), so keep the first.
cohort = (
    decedents_all_carts
    .drop(columns=["source_cart"])
    .drop_duplicates(subset="id", keep="first")
    .set_index("id")
    .join(carts_per_person)
    .reset_index()
)

# 3) Turn numeric columns from text into numbers. Blank text becomes NaN (missing).
numeric_columns = [
    "birth_year", "death_year", "age_years", "age_months", "years_in_us",
    "pregnancies", "live_births", "birth_weight", "living_weight",
    "living_height", "cadaver_length", "cadaver_weight", "time_delay_after_death",
]
for column_name in numeric_columns:
    cohort[column_name] = pd.to_numeric(cohort[column_name].str.strip().replace("", pd.NA), errors="coerce")

# 4) Checks.
print("Rows:", len(cohort), "| distinct ids:", cohort["id"].nunique())
print()
print("People by carts they appear in:")
print(cohort["source_carts"].value_counts().to_string())
print()
print(cohort[["age_years", "cadaver_length", "cadaver_weight", "time_delay_after_death", "death_year"]].describe().round(1).to_string())

In [ ]:
# 1) People per death year and cart group.
print("DEATH YEAR x SOURCE CARTS")
print(pd.crosstab(cohort["death_year"], cohort["source_carts"]).to_string())
print()

# 2) The odd rows: missing age, blank label, unknown sex.
odd_rows = cohort[
    cohort["age_years"].isna()
    | (cohort["primary_cause_of_death"].str.strip() == "")
    | (cohort["sex_code"] == "Unknown")
]
print("ODD ROWS")
print(odd_rows[["id", "age_years", "sex_code", "primary_cause_of_death", "source_carts"]].to_string(index=False))
print()

# 3) The 5 largest time_delay_after_death values.
print("LARGEST time_delay_after_death")
print(cohort["time_delay_after_death"].nlargest(5).to_string())
print()

# 4) Are missing cadaver_length and cadaver_weight the same people?
print("Missing both length and weight:", int((cohort["cadaver_length"].isna() & cohort["cadaver_weight"].isna()).sum()))
print("Missing length only:", int((cohort["cadaver_length"].isna() & cohort["cadaver_weight"].notna()).sum()))
print("Missing weight only:", int((cohort["cadaver_length"].notna() & cohort["cadaver_weight"].isna()).sum()))

In [ ]:
# Reason column: blank means "keep".
cohort["exclude_reason"] = ""

blank_label = cohort["primary_cause_of_death"].str.strip() == ""
unknown_sex_missing_age = (cohort["sex_code"] == "Unknown") & cohort["age_years"].isna()

cohort.loc[blank_label, "exclude_reason"] = "blank label"
cohort.loc[unknown_sex_missing_age, "exclude_reason"] = "unknown sex and missing age"

# Flag unusual death years (kept, but flagged for the holdout design).
cohort["unusual_death_year"] = ~cohort["death_year"].between(2011, 2017)

# Save.
PROCESSED_FOLDER = Path("../data/processed")
PROCESSED_FOLDER.mkdir(parents=True, exist_ok=True)
cohort.to_parquet(PROCESSED_FOLDER / "cohort_decedents.parquet", index=False)

print("Saved. Rows:", len(cohort))
print(cohort["exclude_reason"].value_counts().to_string())
print("Unusual death years:", int(cohort["unusual_death_year"].sum()))

In [ ]:
# Stack the 3 carts and drop exact repeated rows (safe, checked in Step 1h).
medical_history_all = pd.concat(
    [all_tables[(cart, "medical-history")] for cart in ["2960", "2966", "2975"]],
    ignore_index=True,
).drop(columns=["source_cart"]).drop_duplicates()

print("Rows after removing repeats:", len(medical_history_all))
print()
print("TYPE x DETAIL (top 40 combinations)")
print(
    medical_history_all
    .groupby(["medical_history_type", "medical_history_detail"])
    .size()
    .sort_values(ascending=False)
    .head(40)
    .to_string()
)
print()
print("All values of medical_history_type:")
print(medical_history_all["medical_history_type"].value_counts().to_string())

In [ ]:
# 1) Keep only Diagnosis rows with real content, with clean text.
diagnosis_rows = medical_history_all[
    medical_history_all["medical_history_type"].str.strip() == "Diagnosis"
].copy()
diagnosis_rows["detail"] = diagnosis_rows["medical_history_detail"].str.strip()

# 2) Which people have ANY Diagnosis row? Those with a row are "recorded" in some way.
people_with_diagnosis_row = set(diagnosis_rows["id"])

# 3) Choose the diagnoses to turn into columns (those with at least 20 people).
people_per_diagnosis = (
    diagnosis_rows[~diagnosis_rows["detail"].isin(["None", "Unknown", ""])]
    .groupby("detail")["id"].nunique()
    .sort_values(ascending=False)
)
common_diagnoses = people_per_diagnosis[people_per_diagnosis >= 20].index.tolist()
print("Common diagnoses (>= 20 people):")
print(people_per_diagnosis[people_per_diagnosis >= 20].to_string())
print()

# 4) Build the table: one row per person in the cohort.
diagnosis_features = pd.DataFrame({"id": cohort["id"]})
for diagnosis_name in common_diagnoses:
    people_with_this = set(diagnosis_rows.loc[diagnosis_rows["detail"] == diagnosis_name, "id"])
    column_name = "dx_" + diagnosis_name.lower().replace(" ", "_").replace("/", "_")

    def state_of(person_id):
        if person_id in people_with_this:
            return "present"
        if person_id in people_with_diagnosis_row:
            return "recorded_none_or_other"   # has a diagnosis record, but not this one
        return "not_recorded"                  # no diagnosis record at all

    diagnosis_features[column_name] = diagnosis_features["id"].map(state_of)

# 5) Check the states for one column.
first_column = [c for c in diagnosis_features.columns if c.startswith("dx_")][0]
print("States of", first_column)
print(diagnosis_features[first_column].value_counts().to_string())
print()
print("People with any Diagnosis row:", len(people_with_diagnosis_row), "of", len(cohort))

In [ ]:
# 1) Add the "has any diagnosis row" flag.
diagnosis_features["has_any_diagnosis_row"] = diagnosis_features["id"].isin(people_with_diagnosis_row)

# 2) Share of people with a diagnosis row, by cart group.
check_table = cohort[["id", "source_carts"]].merge(
    diagnosis_features[["id", "has_any_diagnosis_row"]], on="id"
)
print("Share with any Diagnosis row, by cart group:")
print(
    check_table.groupby("source_carts")["has_any_diagnosis_row"]
    .agg(people="size", share_with_record="mean")
    .round(2)
    .to_string()
)
print()

# 3) Save.
diagnosis_features.to_parquet(PROCESSED_FOLDER / "features_diagnosis.parquet", index=False)
print("Saved features_diagnosis.parquet:", diagnosis_features.shape)

In [ ]:
# Tables to flag, with the columns that hold the real values.
tables_to_flag = {
    "medical_history": ("medical-history", ["medical_history_type", "medical_history_detail"]),
    "medications": ("medications", ["drug_number"]),
    "substance_usage": ("substance-usages", ["substance_usage_type"]),
    "env_history": ("env-history", ["environmental_factor_type", "environmental_factor_item"]),
    "ancestry": ("ancestry", ["ancestor", "birthplace", "city_of_birth", "state_of_birth", "country_of_birth"]),
    "chromosomal": ("chromosomal-abnormalities", ["chromosomal_abnormalities_type", "abnormality_factor", "is_familial"]),
}

record_flags = pd.DataFrame({"id": cohort["id"]})

for flag_name, (table_name, value_columns) in tables_to_flag.items():
    stacked = pd.concat(
        [all_tables[(cart, table_name)] for cart in ["2960", "2966", "2975"]],
        ignore_index=True,
    )
    # A row is real if at least one value column is non-blank.
    has_content = (stacked[value_columns].apply(lambda col: col.str.strip()) != "").any(axis=1)
    people_with_real_row = set(stacked.loc[has_content, "id"])
    record_flags["has_" + flag_name + "_record"] = record_flags["id"].isin(people_with_real_row)

# Share with a record, by cart group (rows = cart group).
flag_columns = [c for c in record_flags.columns if c.startswith("has_")]
by_cart = cohort[["id", "source_carts"]].merge(record_flags, on="id")
print(by_cart.groupby("source_carts")[flag_columns].mean().round(2).to_string())
print()

record_flags.to_parquet(PROCESSED_FOLDER / "features_record_flags.parquet", index=False)
print("Saved features_record_flags.parquet:", record_flags.shape)

In [ ]:
# Substance usage: all values with the number of people.
substance_all = pd.concat(
    [all_tables[(cart, "substance-usages")] for cart in ["2960", "2966", "2975"]],
    ignore_index=True,
).drop(columns=["source_cart"]).drop_duplicates()

print("SUBSTANCE USAGE TYPES (people per value)")
print(substance_all.groupby("substance_usage_type")["id"].nunique().sort_values(ascending=False).to_string())
print()

# Environment history: types and items.
env_all = pd.concat(
    [all_tables[(cart, "env-history")] for cart in ["2960", "2966", "2975"]],
    ignore_index=True,
).drop(columns=["source_cart"]).drop_duplicates()

print("ENV FACTOR TYPES (people per type)")
print(env_all.groupby("environmental_factor_type")["id"].nunique().sort_values(ascending=False).to_string())
print()

print("DRINKING STATUS ITEMS (people per item)")
drinking = env_all[env_all["environmental_factor_type"].str.strip() == "Drinking Status"]
print(drinking.groupby("environmental_factor_item")["id"].nunique().sort_values(ascending=False).to_string())

In [ ]:
# 1) Does any person have more than one Drinking Status value?
drinks_per_person = drinking.groupby("id")["environmental_factor_item"].nunique()
print("People with more than 1 drinking-status value:", int((drinks_per_person > 1).sum()))
print()

# 2) Substance features.
substance_real = substance_all[substance_all["substance_usage_type"].str.strip() != ""].copy()
substance_real["value"] = substance_real["substance_usage_type"].str.strip()
people_with_substance_row = set(substance_real["id"])
people_recorded_none = set(substance_real.loc[substance_real["value"] == "None", "id"])
people_unknown = set(substance_real.loc[substance_real["value"] == "Unknown if used drugs", "id"])
people_with_named_drug = set(
    substance_real.loc[~substance_real["value"].isin(["None", "Unknown if used drugs"]), "id"]
)

def substance_state(person_id):
    if person_id in people_with_named_drug:
        return "present"
    if person_id in people_recorded_none:
        return "recorded_none"
    if person_id in people_unknown:
        return "unknown"
    return "not_recorded"

substance_features = pd.DataFrame({"id": cohort["id"]})
substance_features["substance_any_drug"] = substance_features["id"].map(substance_state)

# 3) Drinking status feature (one value per person, or not_recorded).
drinking_clean = (
    drinking.assign(item=drinking["environmental_factor_item"].str.strip())
    .drop_duplicates(subset="id")
    .set_index("id")["item"]
)
substance_features["drinking_status"] = (
    substance_features["id"].map(drinking_clean).fillna("not_recorded")
)

print(substance_features["substance_any_drug"].value_counts().to_string())
print()
print(substance_features["drinking_status"].value_counts().to_string())

substance_features.to_parquet(PROCESSED_FOLDER / "features_substance_drinking.parquet", index=False)

In [ ]:
multi_drinking_ids = drinks_per_person[drinks_per_person > 1].index

print(
    drinking[drinking["id"].isin(multi_drinking_ids)]
    .sort_values("id")[["id", "environmental_factor_item"]]
    .to_string(index=False)
)

# Which carts are they from?
print()
print(cohort.loc[cohort["id"].isin(multi_drinking_ids), ["id", "source_carts"]].to_string(index=False))

In [ ]:
# Clean the drinking text first.
drinking_clean_rows = drinking.assign(
    item=drinking["environmental_factor_item"].str.strip()
)[["id", "item"]].drop_duplicates()

# Per person: how many distinct values, and what is the single value (if any).
values_per_person = drinking_clean_rows.groupby("id")["item"].agg(["nunique", "first"])

drinking_status_fixed = values_per_person["first"].where(
    values_per_person["nunique"] == 1, "multiple_values"
)

substance_features["drinking_status"] = (
    substance_features["id"].map(drinking_status_fixed).fillna("not_recorded")
)
substance_features["drinking_multiple"] = substance_features["id"].isin(
    values_per_person.index[values_per_person["nunique"] > 1]
)

print(substance_features["drinking_status"].value_counts().to_string())
print("Flagged multiple:", int(substance_features["drinking_multiple"].sum()))

substance_features.to_parquet(PROCESSED_FOLDER / "features_substance_drinking.parquet", index=False)

In [ ]:
# 1) Count features: how many real rows per person (0 if none).
def count_real_rows(table_name, value_columns, new_column_name):
    stacked = pd.concat(
        [all_tables[(cart, table_name)] for cart in ["2960", "2966", "2975"]],
        ignore_index=True,
    ).drop(columns=["source_cart"]).drop_duplicates()
    has_content = (stacked[value_columns].apply(lambda col: col.str.strip()) != "").any(axis=1)
    counts = stacked.loc[has_content].groupby("id").size().rename(new_column_name)
    return counts

medication_count = count_real_rows("medications", ["drug_number"], "medication_count")
substance_row_count = count_real_rows("substance-usages", ["substance_usage_type"], "substance_row_count")

count_features = pd.DataFrame({"id": cohort["id"]})
count_features["medication_count"] = count_features["id"].map(medication_count)
count_features["substance_row_count"] = count_features["id"].map(substance_row_count)

# Keep NaN where the person has NO record (not recorded). Do not fill with 0.
print(count_features[["medication_count", "substance_row_count"]].describe().round(1).to_string())
print()

# 2) Race and ethnicity (descriptive only, for the audit).
distinctions_all = pd.concat(
    [all_tables[(cart, "distinctions")] for cart in ["2960", "2966", "2975"]],
    ignore_index=True,
).drop(columns=["source_cart"]).drop_duplicates()
distinctions_all["race"] = distinctions_all["race"].str.strip()
race_per_person = distinctions_all.groupby("id")["race"].agg(
    lambda values: values.iloc[0] if values.nunique() == 1 else "multiple_values"
)
count_features["race_summary"] = count_features["id"].map(race_per_person).fillna("not_recorded")
print(count_features["race_summary"].value_counts().to_string())
print()

# 3) Master table: join everything on id.
master = (
    cohort
    .merge(diagnosis_features, on="id", how="left")
    .merge(record_flags, on="id", how="left")
    .merge(substance_features, on="id", how="left")
    .merge(count_features, on="id", how="left")
)
print("Master shape:", master.shape, "| distinct ids:", master["id"].nunique())
master.to_parquet(PROCESSED_FOLDER / "master_table.parquet", index=False)

In [ ]:
reloaded = pd.read_parquet(PROCESSED_FOLDER / "master_table.parquet")
print("Reloaded shape:", reloaded.shape)
print("Duplicate ids:", int(reloaded["id"].duplicated().sum()))
print("Excluded rows:", int((reloaded["exclude_reason"] != "").sum()))

# Columns grouped by their first word, to help with the leakage table.
print()
print(list(reloaded.columns))